# FasalMitra: mini prototype (progress demo)
**Federated Deep Learning for Multi-Crop Disease Detection and Farmer Advisory System**

This notebook is a deliberately **small-scale dry run** of the planned pipeline, so we can show real, working progress and get a feel for the results to expect from the full run:

1. Load PlantVillage and explore the class distribution
2. Take a small balanced subset and hold out a **global test set**
3. Split the rest into **5 non-IID clients** (dominant crops per client + small overlap)
4. Build the **MobileNetV2** transfer-learning model
5. Train a **centralized baseline** (all data pooled)
6. Train with **FedAvg** across the 5 clients (raw images never leave a client)
7. Compare both: accuracy, macro-F1 and the full classification report

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Expected time: ~10-15 min (the one-time PlantVillage download via TFDS, roughly 800 MB, is the longest part).

**Honest scope note:** FedAvg here is implemented by hand in TensorFlow so the notebook runs reliably on Colab. The planned project uses the **Flower** framework; this notebook is the preview of what Phase 2 will run. All numbers come from a tiny subset, a frozen backbone and few epochs, so they are indicative only.

## 1. Configuration (all the small-scale knobs live here)

In [ ]:
SEED          = 42
IMG           = 128      # image side (MobileNetV2 accepts 96/128/160/192/224)
N_PER_CLASS   = 80       # images sampled per class (full dataset has ~150-5000 per class)
TEST_FRAC     = 0.20     # global held-out test set, never seen by any client
NUM_CLIENTS   = 5
OVERLAP       = 0.15     # share of each class that "leaks" to other clients (realistic overlap)
BATCH         = 32
CENTRAL_EPOCHS = 5       # centralized baseline epochs
ROUNDS        = 10       # federated rounds
LOCAL_EPOCHS  = 2        # local epochs per client per round
LR            = 1e-3
WEIGHTS       = 'imagenet'   # pretrained backbone

In [ ]:
import os, re, time, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

try:
    import tensorflow_datasets as tfds
except ImportError:
    %pip install -q tensorflow-datasets
    import tensorflow_datasets as tfds

warnings.filterwarnings('ignore')
tf.keras.utils.set_random_seed(SEED)
os.makedirs('results', exist_ok=True)
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU') or 'none (will be slower)')

## 2. Load PlantVillage and explore it

In [ ]:
# --- TFDS-specific: download dataset + class names + full class counts ---
ds_builder = tfds.builder('plant_village')
ds_builder.download_and_prepare()
class_names = ds_builder.info.features['label'].names
NUM_CLASSES = len(class_names)
print('Total images:', ds_builder.info.splits['train'].num_examples, '| classes:', NUM_CLASSES)

try:  # fast label-only pass (skips image decoding)
    lab_ds = ds_builder.as_dataset(
        split='train', as_supervised=True,
        decoders={'image': tfds.decode.SkipDecoding()}).map(lambda x, y: y)
    full_counts = np.bincount(np.fromiter(lab_ds.as_numpy_iterator(), dtype=np.int64),
                              minlength=NUM_CLASSES)
except Exception as e:
    print('Could not count full-dataset labels quickly:', e)
    full_counts = None

In [ ]:
def crop_of(name):                      # 'Tomato___Late_blight' -> 'Tomato'
    return re.split(r'_{2,}', name)[0]

crops = [crop_of(n) for n in class_names]
short_names = [n.replace('___', ' | ').replace('_', ' ')[:42] for n in class_names]
print(f'{NUM_CLASSES} classes across {len(set(crops))} crops:', sorted(set(crops)))

if full_counts is not None:
    order = np.argsort(full_counts)
    cmap = plt.get_cmap('tab20')
    crop_ids = {c: i for i, c in enumerate(sorted(set(crops)))}
    plt.figure(figsize=(9, 10))
    plt.barh([short_names[i] for i in order], full_counts[order],
             color=[cmap(crop_ids[crops[i]] % 20) for i in order])
    plt.xlabel('Images'); plt.title('PlantVillage: full-dataset class distribution (coloured by crop)')
    plt.tight_layout(); plt.savefig('results/01_class_distribution.png', dpi=150); plt.show()

### 2b. Draw a small, balanced subset (this keeps the whole demo fast)

In [ ]:
def to_small(img, lbl):
    img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(tf.round(tf.clip_by_value(img, 0, 255)), tf.uint8), lbl

stream = (ds_builder.as_dataset(split='train', as_supervised=True)
          .map(to_small, num_parallel_calls=tf.data.AUTOTUNE)
          .shuffle(2000, seed=SEED).prefetch(tf.data.AUTOTUNE))

t0 = time.time()
X_list, y_list = [], []
counts = np.zeros(NUM_CLASSES, dtype=int)
for img, lbl in stream.as_numpy_iterator():
    if counts[lbl] < N_PER_CLASS:
        X_list.append(img); y_list.append(int(lbl)); counts[lbl] += 1
        if counts.min() >= N_PER_CLASS:
            break
X, y = np.stack(X_list), np.array(y_list)
print(f'Subset: {X.shape} | {N_PER_CLASS} per class | sampled in {time.time()-t0:.0f}s')

In [ ]:
# a few sample images (good for a slide)
rng_vis = np.random.default_rng(SEED)
pick = rng_vis.choice(len(y), 8, replace=False)
fig, axs = plt.subplots(2, 4, figsize=(11, 6))
for ax, i in zip(axs.ravel(), pick):
    ax.imshow(X[i]); ax.set_title(short_names[y[i]], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig('results/02_sample_images.png', dpi=150); plt.show()

## 3. Global test set + non-IID split into 5 clients
* A stratified **global test set** is held out *first*, so no client ever trains on it.
* Each class has an **owner client** (clients own different crops, so data is non-IID).
* `OVERLAP` of every class is scattered to the other clients, so clients are skewed but not completely disjoint.

In [ ]:
all_idx = np.arange(len(y))
train_idx, test_idx = train_test_split(all_idx, test_size=TEST_FRAC, stratify=y, random_state=SEED)
X_test, y_test = X[test_idx], y[test_idx]
X_train, y_train = X[train_idx], y[train_idx]
print('Pooled train:', X_train.shape, '| Global test:', X_test.shape)

# crops owned by each client (regional crop variation)
CLIENT_CROPS = {
    0: ['tomato'],
    1: ['apple', 'cherry', 'peach'],
    2: ['corn', 'grape'],
    3: ['potato', 'pepper', 'strawberry'],
    4: ['blueberry', 'orange', 'raspberry', 'soybean', 'squash'],
}
crop2client = {k: c for c, ks in CLIENT_CROPS.items() for k in ks}

def owner_of(cls):
    name = crops[cls].lower()
    for key, c in crop2client.items():
        if name.startswith(key):
            return c
    print('  (unmatched crop, assigned to last client):', crops[cls])
    return NUM_CLIENTS - 1

class_owner = np.array([owner_of(c) for c in range(NUM_CLASSES)])

rng = np.random.default_rng(SEED)
client_idx = {c: [] for c in range(NUM_CLIENTS)}
for cls in range(NUM_CLASSES):
    idxs = train_idx[y[train_idx] == cls].copy()
    rng.shuffle(idxs)
    n_keep = int(round((1 - OVERLAP) * len(idxs)))
    owner = class_owner[cls]
    client_idx[owner] += list(idxs[:n_keep])
    others = [c for c in range(NUM_CLIENTS) if c != owner]
    for i in idxs[n_keep:]:
        client_idx[rng.choice(others)].append(i)

client_data = {}
for c in range(NUM_CLIENTS):
    idx = np.array(client_idx[c]); rng.shuffle(idx)
    client_data[c] = (X[idx], y[idx])
    own = np.isin(y[idx], np.where(class_owner == c)[0]).mean()
    print(f'Client {c+1}: {len(idx):4d} images | {len(set(y[idx])):2d} classes | {own:.0%} from its own crops')

In [ ]:
rows = []
for c in range(NUM_CLIENTS):
    s = pd.Series([crops[k] for k in client_data[c][1]]).value_counts()
    rows.append(s.rename(f'Client {c+1}'))
dist = pd.DataFrame(rows).fillna(0).astype(int)
dist.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab20', width=0.7)
plt.xticks(rotation=0); plt.ylabel('Training images')
plt.title('Non-IID split: images per crop on each client')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.savefig('results/03_noniid_client_split.png', dpi=150); plt.show()
dist

## 4. Model: MobileNetV2 (ImageNet-pretrained, frozen) + custom classifier head

In [ ]:
def make_model():
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=WEIGHTS)
    base.trainable = False
    inp = layers.Input(shape=(IMG, IMG, 3))
    x = layers.Rescaling(1. / 127.5, offset=-1)(inp)           # MobileNetV2 expects [-1, 1]
    x = layers.RandomFlip('horizontal_and_vertical')(x)        # augmentation (training only)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inp, out)

def compile_model(m):
    m.compile(optimizer=tf.keras.optimizers.Adam(LR),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

def evaluate(m, X_, y_):
    pred = m.predict(X_, batch_size=64, verbose=0).argmax(1)
    labels = np.arange(NUM_CLASSES)
    return (pred,
            accuracy_score(y_, pred),
            f1_score(y_, pred, labels=labels, average='macro', zero_division=0),
            f1_score(y_, pred, labels=labels, average='weighted', zero_division=0))

model_preview = make_model()
model_preview.summary()
trainable = sum(int(np.prod(w.shape)) for w in model_preview.trainable_weights)
print(f'\nTrainable parameters (the part exchanged in federated rounds): {trainable:,}')

## 5. Centralized baseline (all training data pooled on one machine)

In [ ]:
t0 = time.time()
central = make_model(); compile_model(central)
central.fit(X_train, y_train, epochs=CENTRAL_EPOCHS, batch_size=BATCH, verbose=2)
pred_c, acc_c, f1m_c, f1w_c = evaluate(central, X_test, y_test)
t_central = time.time() - t0
print(f'\nCENTRALIZED -> test accuracy {acc_c:.3f} | macro-F1 {f1m_c:.3f} | {t_central:.0f}s')

In [ ]:
rep_c = classification_report(y_test, pred_c, labels=np.arange(NUM_CLASSES),
                              target_names=short_names, zero_division=0, digits=3)
print(rep_c)
open('results/classification_report_centralized.txt', 'w').write(rep_c)

## 6. Federated training with FedAvg (5 non-IID clients)
Each round: the server sends the global weights → every client trains locally on **its own images only** → clients send back **weights only** → the server averages them, weighted by client dataset size.

In [ ]:
def get_w(m): return [w.numpy() for w in m.trainable_weights]

def set_w(m, ws):
    for v, w in zip(m.trainable_weights, ws):
        v.assign(w)

def fedavg(updates, sizes):
    tot = float(sum(sizes))
    return [sum(u[i] * (s / tot) for u, s in zip(updates, sizes)) for i in range(len(updates[0]))]

global_model = make_model(); compile_model(global_model)
client_model = make_model()           # reused as the "local trainer" for each client
client_model.set_weights(global_model.get_weights())   # identical frozen backbone everywhere

history, t0 = [], time.time()
for r in range(1, ROUNDS + 1):
    gw = get_w(global_model)
    updates, sizes = [], []
    for c in range(NUM_CLIENTS):
        Xc, yc = client_data[c]
        set_w(client_model, gw)                     # receive global weights
        compile_model(client_model)                 # fresh optimizer each round
        client_model.fit(Xc, yc, epochs=LOCAL_EPOCHS, batch_size=BATCH, verbose=0)
        updates.append(get_w(client_model)); sizes.append(len(yc))   # send weights only
    set_w(global_model, fedavg(updates, sizes))
    _, acc, f1m, _ = evaluate(global_model, X_test, y_test)
    history.append({'round': r, 'accuracy': acc, 'macro_f1': f1m})
    print(f'Round {r:2d}/{ROUNDS} | global test accuracy {acc:.3f} | macro-F1 {f1m:.3f}')

pred_f, acc_f, f1m_f, f1w_f = evaluate(global_model, X_test, y_test)
t_fed = time.time() - t0
print(f'\nFEDAVG -> test accuracy {acc_f:.3f} | macro-F1 {f1m_f:.3f} | {t_fed:.0f}s')

In [ ]:
rep_f = classification_report(y_test, pred_f, labels=np.arange(NUM_CLASSES),
                              target_names=short_names, zero_division=0, digits=3)
print(rep_f)
open('results/classification_report_fedavg.txt', 'w').write(rep_f)

## 7. Federated vs centralized: what to expect

In [ ]:
hist = pd.DataFrame(history)
plt.figure(figsize=(8, 4.5))
plt.plot(hist['round'], hist['accuracy'], marker='o', label='FedAvg: global test accuracy')
plt.axhline(acc_c, color='gray', linestyle='--', label=f'Centralized baseline ({acc_c:.2f})')
plt.xlabel('Federated round'); plt.ylabel('Accuracy'); plt.ylim(0, 1.02); plt.grid(alpha=.3)
plt.title('FedAvg convergence vs centralized baseline'); plt.legend()
plt.tight_layout(); plt.savefig('results/04_fedavg_convergence.png', dpi=150); plt.show()

summary = pd.DataFrame({
    'Method': ['Centralized (pooled data)', f'FedAvg ({NUM_CLIENTS} non-IID clients)'],
    'Accuracy': [acc_c, acc_f],
    'Macro-F1': [f1m_c, f1m_f],
    'Weighted-F1': [f1w_c, f1w_f],
    'Train time (s)': [round(t_central), round(t_fed)],
}).round(3)
summary.to_csv('results/summary.csv', index=False)
hist.to_csv('results/fedavg_per_round.csv', index=False)

# communication cost per round: only trainable weights travel (float32 = 4 bytes)
mb_per_round = trainable * 4 * 2 * NUM_CLIENTS / 1e6      # up + down, all clients
print(f'Communication: ~{mb_per_round:.1f} MB per round for {NUM_CLIENTS} clients '
      f'({trainable:,} trainable parameters), versus uploading every raw image to a server.')
summary

In [ ]:
# bundle everything (charts, reports, tables) for slides; download from Colab
shutil.make_archive('fasalmitra_results', 'zip', 'results')
try:
    from google.colab import files
    files.download('fasalmitra_results.zip')
except Exception:
    print('Results saved in the results/ folder (left sidebar -> Files).')

## 8. How to read and present these results
* **What this proves:** the data pipeline, non-IID client split, MobileNetV2 model, FedAvg loop and evaluation all run end to end.
* **Expect the full run to improve** because it uses far more images per class, a longer schedule and fine-tuning of the top backbone layers. Treat the numbers above as a floor, not the final result.
* **Federated is usually a little below centralized**, especially with skewed clients. That gap is the finding the project will measure and then narrow in the evaluation phase.
* **Small test set:** each class has only a handful of test images, so per-class scores in the report are noisy. Use accuracy and macro-F1 as the headline, not individual rows.
* **Known limitation to mention:** PlantVillage contains near-duplicate images of the same leaf, which can inflate accuracy. It is also lab-style imagery, so real field photos will be harder.
* **Next:** replace the hand-written FedAvg with Flower, run the full dataset, add the centralized fine-tuning stage, then the advisory module.